# Thematische Analyse
Kongruenz nach Thema, Abstimmungstyp und Zeit.

In [ ]:
# Pakete laden und Datensatz einlesen
import pandas as pd
import numpy as np
%load_ext autoreload
%autoreload 2
from visualisierungen import *

# Datensatz mit berechneten Positionen laden – Output von 2_berechnung.ipynb
df = pd.read_csv("../data/processed/df_with_positions.csv")
print(df.info(50))


## 3. Kongruenz nach Thema (`hauptgruppe`)

In [ ]:
# Erster Blick: gibt es Unterschiede in der Zustimmung je nach Themenbereich?
boxplot(df, x="hauptgruppe", y="zustimmung_br-pos",
        titel="Zustimmung nach Hauptgruppe",
        xlabel="Hauptgruppe", ylabel="Zustimmung",
        figsize=(10, 5), rotation=20)

In [ ]:
# Alle relevanten Bundesebenen-Spalten zusammenfassen
bund_cols = ["zustimmung_br-pos", "zustimmung_p-svp", "zustimmung_p-fdp", "zustimmung_p-mitte",
            "zustimmung_p-sps", "zustimmung_p-gps"]

# In Langformat bringen damit alle Akteure auf einer Achse geplottet werden können
df_long = df[["hauptgruppe"] + bund_cols].melt(
    id_vars="hauptgruppe", var_name="bund", value_name="zustimmung")

# Spaltenpräfix entfernen für lesbarere Labels
df_long['bund'] = df_long['bund'].str.replace('zustimmung_', '')

boxplot(df_long, x="hauptgruppe", y="zustimmung", hue="bund", palette = PALETTE_KATEGORIAL_VIELE_WERTE,
        titel="Zustimmung Bundesebene",
        xlabel="Bund", ylabel="Zustimmung", hline = 0, width = 0.8,
        figsize=(30, 20))

In [ ]:
bund_cols = ["zustimmung_br-pos", "zustimmung_p-svp", "zustimmung_p-fdp", "zustimmung_p-mitte",
            "zustimmung_p-sps", "zustimmung_p-gps"]
df['jahr'].astype(int)
# Nur die letzten Jahre anschauen – aktuelle Verhältnisse interessieren uns mehr
df2 = df[df['jahr'] >= 2010]
df_long_2 = df2[["hauptgruppe"] + bund_cols].melt(
    id_vars="hauptgruppe", var_name="bund", value_name="zustimmung")

df_long_2['bund'] = df_long_2['bund'].str.replace('zustimmung_', '')

boxplot(df_long_2, x="bund", y="zustimmung", hue="hauptgruppe", palette = PALETTE_KATEGORIAL_VIELE_WERTE,
        titel="Zustimmung Bundesebene",
        xlabel="Bund", ylabel="Zustimmung", hline = 0, width = 0.8,
        figsize=(30, 20))

In [ ]:
# Zeitliche Dimension hinzufügen – wie hat sich die Zustimmung nach Thema entwickelt?
df_long = df[["jahrzehnt", "hauptgruppe"] + bund_cols].melt(
    id_vars=["jahrzehnt", "hauptgruppe"], var_name="bund", value_name="zustimmung")

df_long['bund'] = df_long['bund'].str.replace('zustimmung_', '')

scatterplot(df_long, x="jahrzehnt", y="zustimmung", hue="bund", palette = PALETTE_KATEGORIAL_VIELE_WERTE,
        titel="Zustimmung Bundesebene",
        xlabel="Bund", ylabel="Zustimmung",
        figsize=(30, 20))

In [ ]:
# Heatmap: Durchschnittliche Zustimmung pro Thema und Akteur – guter Überblick auf einen Blick
pivot = df_long.groupby(["hauptgruppe", "bund"])["zustimmung"].mean().unstack()

heatmap(pivot, xlabel="Akteur", ylabel="Hauptgruppe",
        cmap=CMAP_DIVERGIEREND, fmt=".2f", figsize=(12, 8), rotation=0)

In [ ]:
plt.close('all')

hauptgruppen = sorted(df_long['hauptgruppe'].dropna().unique())
n_groups = len(hauptgruppen)

# Ein Subplot pro Themenbereich – sharex/sharey damit die Achsen vergleichbar bleiben
fig, axes = plt.subplots(n_groups, 1, figsize=(12, 5 * n_groups), sharex=True, sharey=True)

for ax, gruppe in zip(axes, hauptgruppen):
    subset = df_long[df_long['hauptgruppe'] == gruppe]

    # Mittelwert und Anzahl separat berechnen, dann mergen
    means = subset.groupby(["jahrzehnt", "bund"])["zustimmung"].mean().reset_index()
    counts = subset.groupby(["jahrzehnt", "bund"]).size().reset_index(name='n')
    means = means.merge(counts, on=["jahrzehnt", "bund"])

    sns.lineplot(data=means, x="jahrzehnt", y="zustimmung", hue="bund",
                 marker="o", palette=PALETTE_KATEGORIAL_VIELE_WERTE[:4], ax=ax)

    # Wert + n direkt an den Punkten annotieren
    for _, row in means.iterrows():
        ax.annotate(f"{row['zustimmung']:.1f}\n(n={row['n']})",
                    xy=(row['jahrzehnt'], row['zustimmung']),
                    xytext=(0, 10), textcoords='offset points',
                    ha='center', fontsize=7, color='#444444')

    # Gesamt-N pro Jahrzehnt als fettgedruckte Zahl oben drüber
    n_pro_jahrzehnt = subset.groupby('jahrzehnt').size().reset_index(name='n_total')
    y_pos = means['zustimmung'].max() + 0.15

    for _, row in n_pro_jahrzehnt.iterrows():
        ax.annotate(f"N={row['n_total']}",
                    xy=(row['jahrzehnt'], y_pos),
                    ha='center', fontsize=9, color='#222222',
                    fontweight='bold')

    ax.axhline(0, color="#888888", linestyle="--", linewidth=1, alpha=0.7)
    ax.set_title(gruppe, fontsize=14)
    ax.set_ylabel("Zustimmung")
    ax.tick_params(axis='x', labelbottom=True, rotation=45)
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)

axes[-1].set_xlabel("Jahrzehnt")
plt.tight_layout()
plt.show()

In [ ]:
# Nur Parteispalten – BR/BV weglassen damit der Vergleich zwischen Parteien klarer wird
zust_cols = [c for c in df.columns if c.startswith("zustimmung_p-")]
df_long = df.melt(id_vars=["hauptgruppe"], value_vars=zust_cols,
                   var_name="partei", value_name="zustimmung").dropna()
df_long["partei"] = df_long["partei"].str.replace("zustimmung_", "")

parteien_auswahl = ["p-svp", "p-fdp", "p-mitte", "p-sps", "p-gps", "p-glp"]
sub = df_long[df_long["partei"].isin(parteien_auswahl)]
hgs = sub["hauptgruppe"].unique()

# Dotplot mit IQR-Fehlerbalken – übersichtlicher als Boxplot wenn viele Themen auf einmal
fig, axes = plt.subplots(4, 3, figsize=(16, 14), sharex=True)
axes = axes.flatten()

for i, hg in enumerate(hgs):
    ax = axes[i]
    s = sub[sub["hauptgruppe"] == hg]
    stats = s.groupby("partei")["zustimmung"].agg(["median", lambda x: x.quantile(0.25), lambda x: x.quantile(0.75)])
    stats.columns = ["median", "q25", "q75"]
    stats = stats.loc[stats.index.isin(parteien_auswahl)].sort_values("median")

    for j, (p, row) in enumerate(stats.iterrows()):
        ax.errorbar(row["median"], j,
                     xerr=[[row["median"]-row["q25"]], [row["q75"]-row["median"]]],
                     fmt="o", capsize=4, markersize=6)
    ax.axvline(0, color="grey", ls="--", alpha=0.5)
    ax.set_yticks(range(len(stats)))
    ax.set_yticklabels(stats.index, fontsize=8)
    ax.set_title(hg, fontsize=9, fontweight="bold")
    ax.grid(axis="x", alpha=0.3)

# Überzählige Subplots ausblenden
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

# Final Plot
Due to the partially low numbers of initatives to a topic, we have decided to cluster them according to the epoches of the time analysis.

In [ ]:
# Zeitphasen-Mapping für die interaktiven Grafiken
# None = keine Filterung (gesamte Periode), sonst muss der String mit df['phase'] übereinstimmen
zeit_spalten = {
    'Gesamte Zeitperiode':       None,
    'Frühphase (bis 1899)':      'phase1_fruehphase',
    'Volatile Phase (1900–1949)': 'phase2_volatile',
    'Konsensphase (1950–1975)':  'phase3_konsens',
    'Aufspaltung (1976–2009)':   'phase4_aufspaltung',
    'Divergenz (2010-heute)':                   'phase5_2010_heute',
}

# First trial
Problem plot is too crowded

In [ ]:
akteur_cols = [
    'zustimmung_br-pos', 'zustimmung_bv-pos',
    'zustimmung_p-gps',    # Grüne
    'zustimmung_p-sps',    # SP
    'zustimmung_p-mitte',  # Mitte
    'zustimmung_p-fdp',    # FDP
    'zustimmung_p-svp',    # SVP
]

label_map = {
    'zustimmung_br-pos':  'Bundesrat',
    'zustimmung_bv-pos':  'Bundesversammlung',
    'zustimmung_p-gps':   'Grüne',
    'zustimmung_p-sps':   'SP',
    'zustimmung_p-mitte': 'Mitte',
    'zustimmung_p-fdp':   'FDP',
    'zustimmung_p-svp':   'SVP',
}

farben_map = {
    'Bundesrat':         '#4477AA',
    'Bundesversammlung': '#CC6677',
    'Grüne':  '#999933',
    'SP':     '#882255',
    'Mitte':  '#DDCC77',
    'FDP':    '#332288',
    'SVP':    '#117733',
}

fig = boxplot_interaktiv_zeitwahl(
    df,
    wert_cols=akteur_cols,
    hauptgruppe_spalte='hauptgruppe',       # neuer Pflichtparameter
    phasen_spalte='phase',
    zeit_spalten=zeit_spalten,
    label_map=label_map,
    farben_map=farben_map,
    titel='Übereinstimmung pro Akteur und Hauptgruppe',
    xlabel='Hauptgruppe',
    ylabel='Übereinstimmung',
    yrange=(-0.5, 0.5),
)

fig.show()                           # schliessende Klammer ergänzt

## 2nd Trial: Putting BR und BV together

In [ ]:
# Bundesrat und Bundesversammlung zu einer gemeinsamen Bund-Position zusammenfassen
# – macht den Plot lesbarer wenn man sie nicht einzeln braucht
df['zustimmung_bund-pos'] = df[['zustimmung_br-pos', 'zustimmung_bv-pos']].mean(axis=1)

fig = boxplot_interaktiv_zeitwahl(
    df,
    wert_cols=akteur_cols,
    hauptgruppe_spalte='hauptgruppe',
    phasen_spalte='phase',
    zeit_spalten=zeit_spalten,
    label_map=label_map,
    farben_map=farben_map,
    titel='Übereinstimmung pro Akteur und Hauptgruppe',
    xlabel='Hauptgruppe',
    ylabel='Übereinstimmung',
    yrange=(-0.5, 0.5),
)

fig.show()

## 3rd Trial: Facette

In [ ]:
# Endgültige Version: Facetten nach Thema – jeder Themenbereich bekommt seinen eigenen Subplot
# Kürzere Labels (BR, BV statt ausgeschrieben) damit die Legende nicht zu gross wird
akteur_cols = [
    'zustimmung_br-pos',
    'zustimmung_bv-pos',
    'zustimmung_p-gps',
    'zustimmung_p-sps',
    'zustimmung_p-mitte',
    'zustimmung_p-fdp',
    'zustimmung_p-svp',
]

label_map = {
    'zustimmung_br-pos':   'BR',
    'zustimmung_bv-pos':   'BV',
    'zustimmung_p-gps':    'Grüne',
    'zustimmung_p-sps':    'SP',
    'zustimmung_p-mitte':  'Mitte',
    'zustimmung_p-fdp':    'FDP',
    'zustimmung_p-svp':    'SVP',
}

# Farben angelehnt an die offiziellen Parteifarben (ungefähr)
farben_map = {
    'BR':         '#4477AA',
    'BV': '#CC6677',
    'Grüne':  '#999933',
    'SP':     '#882255',
    'Mitte':  '#DDCC77',
    'FDP':    '#332288',
    'SVP':    '#117733',
}

# Reihenfolge der Themen – inhaltlich geordnet, nicht alphabetisch
hauptgruppe_reihenfolge = [
    'Staatsordnung',
    'Aussenpolitik',
    'Sicherheitspolitik',
    'Wirtschaft',
    'Landwirtschaft',
    'Öffentliche Finanzen',
    'Energie',
    'Verkehr & Infrastruktur',
    'Umwelt & Lebensraum',
    'Sozial- und Gesellschaftspolitik',
    'Bildung & Forschung',
    'Kultur, Religion & Medien',
]

# Facettierten Boxplot erstellen und direkt als HTML für den Blog speichern
fig = boxplot_facetiert_zeitwahl(
    df,
    wert_cols=akteur_cols,
    hauptgruppe_spalte='hauptgruppe',
    hauptgruppe_reihenfolge=hauptgruppe_reihenfolge,
    phasen_spalte='phase',
    zeit_spalten=zeit_spalten,
    label_map=label_map,
    farben_map=farben_map,
    titel=None,
    ylabel='Übereinstimmung',
    yrange=(-0.5, 0.5),
    n_cols=4,
)

fig.show()

fig.write_html(
    "../Blog/blog_plots/d7_thema_facetiert.html",
    include_plotlyjs='inline',
    full_html=True,
)

In [ ]:
df[(df['hauptgruppe']=='Staatsordnung') & (df['zustimmung_p-svp']<0)]

In [ ]:
import math
import plotly.graph_objects as go


def mean_tabelle_zeitwahl(
    df,
    wert_cols,
    hauptgruppe_spalte,
    phasen_spalte,
    zeit_spalten,
    hauptgruppe_reihenfolge=None,
    label_map=None,
    farben_map=None,
    default_label='Gesamte Zeitperiode',
    titel="",
    thema_kopf="Themenbereich",
    rundung=2,
    neg_farbe="#B23A48",
    pos_farbe="#222222",
    leer_text="",
    hoehe=None,
):
    label_map = label_map or {}
    farben_map = farben_map or {}

    if hauptgruppe_reihenfolge is None:
        hauptgruppen = df[hauptgruppe_spalte].dropna().unique().tolist()
    else:
        hauptgruppen = list(hauptgruppe_reihenfolge)

    akteur_namen = [label_map.get(c, c) for c in wert_cols]
    n_rows = len(hauptgruppen)

    def _kontrast_text(hexfarbe):
        h = (hexfarbe or "#888888").lstrip('#')
        r, g, b = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
        luminanz = 0.299 * r + 0.587 * g + 0.114 * b
        return '#000000' if luminanz > 150 else '#FFFFFF'

    def _fmt(v):
        if v is None or (isinstance(v, float) and math.isnan(v)):
            return leer_text, False
        v = round(float(v), rundung)
        if v == 0:
            v = 0.0
        return f"{v:.{rundung}f}", v < 0

    # Kopfzeile: Themenspalte neutral, Akteurspalten in Parteifarben
    kopf_werte = [f"<b>{thema_kopf}</b>"] + [f"<b>{n}</b>" for n in akteur_namen]
    kopf_bg = ["#3A3A3A"] + [farben_map.get(n, "#888888") for n in akteur_namen]
    kopf_text = ["#FFFFFF"] + [_kontrast_text(farben_map.get(n)) for n in akteur_namen]

    # Zebra-Streifen pro Zeile, für jede Spalte dasselbe Muster
    streifen = [
        "rgba(0,0,0,0.05)" if i % 2 else "rgba(0,0,0,0.0)"
        for i in range(n_rows)
    ]

    breiten = [2.4] + [1.0] * len(wert_cols)
    labels = list(zeit_spalten.keys())

    fig = go.Figure()

    for lab in labels:
        phase = zeit_spalten[lab]
        df_subset = df if phase is None else df[df[phasen_spalte] == phase]

        spalten_werte = [list(hauptgruppen)]
        zell_textfarben = [["#222222"] * n_rows]  # Themenspalte immer dunkel
        for col in wert_cols:
            werte, farben = [], []
            for hg in hauptgruppen:
                sub = df_subset[df_subset[hauptgruppe_spalte] == hg]
                txt, ist_neg = _fmt(sub[col].mean())
                werte.append(txt)
                farben.append(neg_farbe if ist_neg else pos_farbe)
            spalten_werte.append(werte)
            zell_textfarben.append(farben)

        fig.add_trace(go.Table(
            columnwidth=breiten,
            header=dict(
                values=kopf_werte,
                fill_color=kopf_bg,
                font=dict(color=kopf_text, size=13),
                align=['left'] + ['center'] * len(wert_cols),
                line_color="rgba(0,0,0,0.12)",
                height=36,
            ),
            cells=dict(
                values=spalten_werte,
                fill_color=[streifen] * (len(wert_cols) + 1),
                font=dict(color=zell_textfarben, size=13),
                align=['left'] + ['center'] * len(wert_cols),
                line_color="rgba(0,0,0,0.08)",
                height=28,
            ),
            visible=(lab == default_label),
        ))

    n_labels = len(labels)
    buttons = [
        dict(
            label=lab,
            method='update',
            args=[{'visible': [j == i for j in range(n_labels)]}],
        )
        for i, lab in enumerate(labels)
    ]

    if hoehe is None:
        hoehe = 28 * n_rows + 190

    fig.update_layout(
        title=titel,
        template=PLOTLY_TEMPLATE,
        font=dict(family=PLOTLY_FONT_FAMILY),
        paper_bgcolor=PLOTLY_BG_TRANSPARENT,
        height=hoehe,
        margin=dict(t=120, b=20, l=10, r=10),
        updatemenus=[dict(
            type="buttons",
            direction="right",
            x=0.5, xanchor="center",
            y=1.0, yanchor="bottom",
            buttons=buttons,
            showactive=True,
            active=labels.index(default_label) if default_label in labels else 0,
        )],
    )

    return fig

fig_mean= mean_tabelle_zeitwahl(
    df,
    wert_cols=akteur_cols,
    hauptgruppe_spalte='hauptgruppe',
    hauptgruppe_reihenfolge=hauptgruppe_reihenfolge,
    phasen_spalte='phase',
    zeit_spalten=zeit_spalten,
    label_map=label_map,
    farben_map=farben_map,
    titel=None,
    rundung=2,
)

fig_mean.show()


In [ ]:
import math
import plotly.graph_objects as go


def median_tabelle_zeitwahl(
    df,
    wert_cols,
    hauptgruppe_spalte,
    phasen_spalte,
    zeit_spalten,
    hauptgruppe_reihenfolge=None,
    label_map=None,
    farben_map=None,
    default_label='Gesamte Zeitperiode',
    titel="",
    thema_kopf="Themenbereich",
    rundung=2,
    neg_farbe="#B23A48",
    pos_farbe="#222222",
    leer_text="",
    hoehe=None,
):
    label_map = label_map or {}
    farben_map = farben_map or {}

    if hauptgruppe_reihenfolge is None:
        hauptgruppen = df[hauptgruppe_spalte].dropna().unique().tolist()
    else:
        hauptgruppen = list(hauptgruppe_reihenfolge)

    akteur_namen = [label_map.get(c, c) for c in wert_cols]
    n_rows = len(hauptgruppen)

    def _kontrast_text(hexfarbe):
        h = (hexfarbe or "#888888").lstrip('#')
        r, g, b = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
        luminanz = 0.299 * r + 0.587 * g + 0.114 * b
        return '#000000' if luminanz > 150 else '#FFFFFF'

    def _fmt(v):
        if v is None or (isinstance(v, float) and math.isnan(v)):
            return leer_text, False
        v = round(float(v), rundung)
        if v == 0:
            v = 0.0
        return f"{v:.{rundung}f}", v < 0

    # Kopfzeile: Themenspalte neutral, Akteurspalten in Parteifarben
    kopf_werte = [f"<b>{thema_kopf}</b>"] + [f"<b>{n}</b>" for n in akteur_namen]
    kopf_bg = ["#3A3A3A"] + [farben_map.get(n, "#888888") for n in akteur_namen]
    kopf_text = ["#FFFFFF"] + [_kontrast_text(farben_map.get(n)) for n in akteur_namen]

    # Zebra-Streifen pro Zeile, für jede Spalte dasselbe Muster
    streifen = [
        "rgba(0,0,0,0.05)" if i % 2 else "rgba(0,0,0,0.0)"
        for i in range(n_rows)
    ]

    breiten = [2.4] + [1.0] * len(wert_cols)
    labels = list(zeit_spalten.keys())

    fig = go.Figure()

    for lab in labels:
        phase = zeit_spalten[lab]
        df_subset = df if phase is None else df[df[phasen_spalte] == phase]

        spalten_werte = [list(hauptgruppen)]
        zell_textfarben = [["#222222"] * n_rows]  # Themenspalte immer dunkel
        for col in wert_cols:
            werte, farben = [], []
            for hg in hauptgruppen:
                sub = df_subset[df_subset[hauptgruppe_spalte] == hg]
                txt, ist_neg = _fmt(sub[col].median())
                werte.append(txt)
                farben.append(neg_farbe if ist_neg else pos_farbe)
            spalten_werte.append(werte)
            zell_textfarben.append(farben)

        fig.add_trace(go.Table(
            columnwidth=breiten,
            header=dict(
                values=kopf_werte,
                fill_color=kopf_bg,
                font=dict(color=kopf_text, size=13),
                align=['left'] + ['center'] * len(wert_cols),
                line_color="rgba(0,0,0,0.12)",
                height=36,
            ),
            cells=dict(
                values=spalten_werte,
                fill_color=[streifen] * (len(wert_cols) + 1),
                font=dict(color=zell_textfarben, size=13),
                align=['left'] + ['center'] * len(wert_cols),
                line_color="rgba(0,0,0,0.08)",
                height=28,
            ),
            visible=(lab == default_label),
        ))

    n_labels = len(labels)
    buttons = [
        dict(
            label=lab,
            method='update',
            args=[{'visible': [j == i for j in range(n_labels)]}],
        )
        for i, lab in enumerate(labels)
    ]

    if hoehe is None:
        hoehe = 28 * n_rows + 190

    fig.update_layout(
        title=titel,
        template=PLOTLY_TEMPLATE,
        font=dict(family=PLOTLY_FONT_FAMILY),
        paper_bgcolor=PLOTLY_BG_TRANSPARENT,
        height=hoehe,
        margin=dict(t=120, b=20, l=10, r=10),
        updatemenus=[dict(
            type="buttons",
            direction="right",
            x=0.5, xanchor="center",
            y=1.0, yanchor="bottom",
            buttons=buttons,
            showactive=True,
            active=labels.index(default_label) if default_label in labels else 0,
        )],
    )

    return fig

fig_median = median_tabelle_zeitwahl(
    df,
    wert_cols=akteur_cols,
    hauptgruppe_spalte='hauptgruppe',
    hauptgruppe_reihenfolge=hauptgruppe_reihenfolge,
    phasen_spalte='phase',
    zeit_spalten=zeit_spalten,
    label_map=label_map,
    farben_map=farben_map,
    titel=None,
    rundung=2,
)

fig_median.show()
